# tdlpackio v2.0 Demo

tdlpackio v2.0 is refactored from pytdlpack v1 and heavily borrows its API, code design, and data access methods from [grib2io](https://github.com/NOAA-MDL/grib2io). In pytdlpack v1.x, there was an experimental module, TdlpackIO, that featured some of the new code.  TdlpackIO, along with lessons learned from grib2io v2 development, are foundation for tdlpackio v2. In summary, if you are familiar with grib2io, then tdlpackio will look and feel the same. This notebook will not discuss the design principles around the structure of TDLPACK files or the data format.
In this demo, we will be working with sample TDLPACK files from the tdlpackio repository.

* [hre201701.sq](https://github.com/NOAA-MDL/tdlpackio/blob/master/tests/data/hre201701.sq) - TDLPACK Vector (i.e. station) Sequential File _*(containing hourly METAR observations for Jan. 2017)_*
* [gfspkd47.2017020100.sq](https://github.com/NOAA-MDL/tdlpackio/blob/master/tests/data/gfspkd47.2017020100.sq) - TDLPACK Gridded Sequential File (_*(containing 00Z GFS 47km grids for Jan. 2017)_*
* [blend.analysisgrconst.co.ra](https://github.com/NOAA-MDL/tdlpackio/blob/master/tests/data/blend.analysisgrconst.co.ra) - TDLPACK Gridded Random Access File

**NOTE:** TDLPACK data format does not have official file name extensions, but file extensions you might see are `.ra`, `.sq`, `.grd_ra`, `.grd_sq`, and `.tdlp`.

In [1]:
import tdlpackio
import datetime
import numpy as np

# Opening TDLPACK Files
---

**_NOTE:_** _For this initial example we will focus on reading a TDLPACK sequential file containing gridded records. Later in this demo, we will show the unique features of vector data as well as both types with respect to random-access files. As you will see, and by design, the type of file is does not change the usage of tdlpackio._

Opening an existing TDLPACK file is the same a pytdlpack, but note that `tdlpackio.open()` calls an open class instead of a function.

## TDLPACK Sequential File (Gridded Records)

In [2]:
f = tdlpackio.open("gfspkd47.2017020100.sq")
print(f)

path = gfspkd47.2017020100.sq
mode = rb
format = None
ra_template = None
name = /Users/ericengle/Repos/GitHub-NOAA/tdlpackio/demos/gfspkd47.2017020100.sq
records = 100
filetype = sequential
size = 3363792



### What is tdlpackio doing when opening a TDLPACK file?
* File is opened in Python to determine the TDLPACK file type -- is it sequential or random-access?
* The file is indexed. **IMPORTANT:** The record indexing is performed in Python for both file types. The open class contains indexing methods that only read the TDLPACK record metadata, then skips to the next record.
* At this point, data are not read from the file.
* When a TDLPACK data record is detected, the metadata are unpacked via libtdlpack `unpack_meta` subroutine.

## Accessing TDLPACK Records from file

When a TDLPACK file is opened for reading, technically, all of the TDLPACK records on the file are represented in the open class object via `TdlpackRecord`, `TdlpackStationRecord`, or `TdlpackTrailerRecord` are created through indexing and stored in `f._index['record']`.

The best way to access TDLPACK records is to use index or slice syntax on the open class object.

In [3]:
# Index
rec = f[10]
print(rec)

10:d=2017020100:001000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB HGT GFS


In [4]:
# Slice
for rec in f[5:10]:
    print(rec)

5:d=2017020100:001000008 000000850 000000000 0000000000:  0-HR FCST: 850 MB HGT GFS
6:d=2017020100:001000008 000000800 000000000 0000000000:  0-HR FCST: 800 MB HGT GFS
7:d=2017020100:001000008 000000750 000000000 0000000000:  0-HR FCST: 750 MB HGT GFS
8:d=2017020100:001000008 000000700 000000000 0000000000:  0-HR FCST: 700 MB HGT GFS
9:d=2017020100:001000008 000000600 000000000 0000000000:  0-HR FCST: 600 MB HGT GFS


## Selection of TDLPACK records based on attributes

Use **_any_** TdlpackRecord object attribute name as a keyword for the file object `select()` method. **NOTE:** The `select()` method will always return a list.

#### Example 1: Select V-wind records.

The TDLPACK ID components for V-wind is CCC = 004, FFF = 100

In [5]:
records = f.select(ccc=4,fff=100)
for rec in records:
    print(rec)

57:d=2017020100:004100008 000001000 000000000 0000000000:  0-HR FCST:1000 MB V GRD GFS
58:d=2017020100:004100008 000000975 000000000 0000000000:  0-HR FCST: 975 MB V GRD GFS
59:d=2017020100:004100008 000000950 000000000 0000000000:  0-HR FCST: 950 MB V GRD GFS
60:d=2017020100:004100008 000000925 000000000 0000000000:  0-HR FCST: 925 MB V GRD GFS
61:d=2017020100:004100008 000000900 000000000 0000000000:  0-HR FCST: 900 MB V GRD GFS
62:d=2017020100:004100008 000000850 000000000 0000000000:  0-HR FCST: 850 MB V GRD GFS
63:d=2017020100:004100008 000000800 000000000 0000000000:  0-HR FCST: 800 MB V GRD GFS
64:d=2017020100:004100008 000000750 000000000 0000000000:  0-HR FCST: 750 MB V GRD GFS
65:d=2017020100:004100008 000000700 000000000 0000000000:  0-HR FCST: 700 MB V GRD GFS
66:d=2017020100:004100008 000000600 000000000 0000000000:  0-HR FCST: 600 MB V GRD GFS
67:d=2017020100:004100008 000000500 000000000 0000000000:  0-HR FCST: 500 MB V GRD GFS
68:d=2017020100:004100008 000000300 0000000

#### Example 2: Select all 500 mb records.

In [6]:
records = f.select(uuuu=500)
for rec in records:
    print(rec)

10:d=2017020100:001000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB HGT GFS
24:d=2017020100:002000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB TEMP GFS
38:d=2017020100:003000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB RH GFS
53:d=2017020100:004000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB U GRD GFS
67:d=2017020100:004100008 000000500 000000000 0000000000:  0-HR FCST: 500 MB V GRD GFS
81:d=2017020100:005000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB VV GFS


#### Example 3: Select records with a primary missing value

In [7]:
records = f.select(primaryMissingValue=9999)
for rec in records:
    print(rec)

93:d=2017020100:003708008 000000010 000000000 0000000000:  0-HR FCST:VOL SOIL MOIST 0-.1M BGL GFS
94:d=2017020100:003708008 000000040 000000000 0000000000:  0-HR FCST:VOL SOIL MOIST .1-.4M BGL GFS
95:d=2017020100:003708008 000000100 000000000 0000000000:  0-HR FCST:VOL SOIL MOIST .4-1.M BGL GFS


# TdlpackRecord Object Metadata Attributes
---

The TdlpackRecord object, attributes, and metadata storage have changed significantly from pytdlpack v1.x and is very similar to grib2io.

The TdlpackRecord class inherits from a TdlpackRecord base class that contains the attributes describing TDLPACK metadata. Since a TDLPACK record can store vector or gridded data, metadata stored in the Grid Definition Section is not always available, and is only present when the TdlpackRecord object instance is created with non-zero IS2 array data or is explicitly stated using the `type = "grid` kwarg. The TdlpackRecord attributes are defined as a `dataclass.field` object and has a default value that is a unique descriptor class with custom implementations of the `__get__` and `__set__` protocols.

In [8]:
rec = f[10]
rec

Section 0: edition = 0
Section 1: sectionFlags = {'hasBitMapSection': 0, 'hasGridDefinitionSection': 1}
Section 1: year = 2017
Section 1: month = 2
Section 1: day = 1
Section 1: hour = 0
Section 1: minute = 0
Section 1: refDate = 2017-02-01 00:00:00
Section 1: id = 001000008 000000500 000000000 0000000000
Section 1: leadTime = 0:00:00
Section 1: leadTimeHours = 0
Section 1: leadTimeMinutes = 0
Section 1: modelID = 8
Section 1: modelSequenceID = 1
Section 1: decScaleFactor = 0
Section 1: binScaleFactor = 0
Section 1: name =  500 MB HGT GFS                 
Section 1: validDate = 2017-02-01 00:00:00
Section 1: duration = 0:00:00
Section 2: mapProjection = 5
Section 2: nx = 297
Section 2: ny = 169
Section 2: latitudeLowerLeft = 2.8320000000000003
Section 2: longitudeLowerLeft = 150.0
Section 2: orientationLongitude = 105.0
Section 2: gridLength = 47625.0
Section 2: standardLatitude = 60.0
Section 2: projParams = {'a': 6371229.0, 'b': 6371229.0, 'proj': 'stere', 'lat_ts': 60.0, 'lat_0': 90

Above is the "print" from the `TdlpackRecord.__repr__`. When the record is printed via `print`, the `__str__` is used and provides an itdlp-like output.

In [9]:
print(rec)

10:d=2017020100:001000008 000000500 000000000 0000000000:  0-HR FCST: 500 MB HGT GFS


The metadata attributes are dynamically mapped from the TDLPACK integer-coded array metadata sections: `is0`, `is1`, `is2`, and `is4`. A complete description of the TDLPACK section arrays can be found in [TDL Office Note 00-1](https://www.weather.gov/media/mdl/TDL_OfficeNote00-1.pdf), Chapter 5.

In [10]:
print(f"IS0: {rec.is0}")
print(f"IS0: {rec.is1}")
print(f"IS0: {rec.is2}")
print(f"IS0: {rec.is4}")

IS0: [1347175508      18905          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0
          0          0          0          0          0          0]
IS0: [        71          1       2017          2          1          0
          0 2017020100    1000008        500          0          0
          0          0          8          1          0          0
          0          0          0         32         32         53
         48         48         32         77         66         32
         72         71         84         32       

The plain language descriptor string supports up to 32 characters and will pad with blank characters.

In [11]:
print(f"{rec.name = }")
print(rec.is1)
rec.name = "HELLO WORLD!"
print(f"{rec.name = }")
print(rec.is1)

rec.name = ' 500 MB HGT GFS                 '
[        71          1       2017          2          1          0
          0 2017020100    1000008        500          0          0
          0          0          8          1          0          0
          0          0          0         32         32         53
         48         48         32         77         66         32
         72         71         84         32         71         70
         83         32         32         32         32         32
         32         32         32         32         32         32
         32         32         32         32         32         32]
rec.name = 'HELLO WORLD!                    '
[        71          1       2017          2          1          0
          0 2017020100    1000008        500          0          0
          0          0          8          1          0          0
          0          0          0         32         72         69
         76         76         79   

### Dealing with Dates and Times

tdlpackio represents all TDLPACK record date and time metadata as `datetime.datetime` or `datetime.timedelta` objects and strictly checks the type when you are setting these metadata attributes. The following table explains the date and time metadata in more detail:

| Attr | Object Type | Mutable | Linked Relationship | Data Array and Location |
| --- | --- | --- | --- | --- |
| `year` | `int` | YES | `refDate` | `is1[2]` |
| `month` | `int` | YES | `refDate` | `is1[3]` |
| `day` | `int` | YES | `refDate` | `is1[4]` |
| `hour` | `int` | YES | `refDate` | `is1[5]` |
| `minute` | `int` | YES | `refDate` | `is1[6]` |
| `refDate` | `datetime.datetime` | YES | `year`, `month`, `day`, `hour`, `minute`, `validDate` | `is1[7]` |
| `leadTime` | `datetime.timedelta` | YES | `leadTimeHours`, `leadTimeMinutes`, `validDate`, `id`, `id.word3`, `id.tau` | None - it is computed when referenced `(leadTimeHours + leadTimeMinutes)` |
| `leadTimeHours` | `datetime.timedelta` | YES | `leadTime`, `validDate` | `is1[10]` (word 3 of ID), `is1[12]` |
| `leadTimeMinutes` | `datetime.timedelta` | YES | `leadTime`, `validDate` | `is1[13]` |
| `validDate` | `datetime.datetime` | NO | None | None - it is computed when referenced `(refDate + leadTime)` |

The `validDate` is the only date/time attribute that is immutable (i.e. it is read-only). The column, **"Linked Relationship"**, answers the question, _"If I change this attribute, what other `TdlpackRecord` attributes will be changed?"_ and column, **"Data Array and Location"**, explains what items in the `is1` array are modified.

In [12]:
print(f"{rec.year = }")
print(f"{rec.month = }")
print(f"{rec.day = }")
print(f"{rec.hour = }")
print(f"{rec.minute = }")
print(f"{rec.refDate = }")
print(f"{rec.leadTimeHours = }")
print(f"{rec.leadTimeMinutes = }")
print(f"{rec.leadTime = }")
print(f"{rec.validDate = }")

rec.year = 2017
rec.month = 2
rec.day = 1
rec.hour = 0
rec.minute = 0
rec.refDate = datetime.datetime(2017, 2, 1, 0, 0)
rec.leadTimeHours = 0
rec.leadTimeMinutes = 0
rec.leadTime = datetime.timedelta(0)
rec.validDate = datetime.datetime(2017, 2, 1, 0, 0)


Lets change some of these attributes and print again...

In [13]:
rec.year = 2026
rec.month = 3
rec.day = 1
rec.hour = 12
rec.minute = 0
rec.leadTimeHours = 240
print(f"{rec.year = }")
print(f"{rec.month = }")
print(f"{rec.day = }")
print(f"{rec.hour = }")
print(f"{rec.minute = }")
print(f"{rec.refDate = }")
print(f"{rec.leadTimeHours = }")
print(f"{rec.leadTimeMinutes = }")
print(f"{rec.leadTime = }")
print(f"{rec.validDate = }")

rec.year = 2026
rec.month = 3
rec.day = 1
rec.hour = 12
rec.minute = 0
rec.refDate = datetime.datetime(2026, 3, 1, 12, 0)
rec.leadTimeHours = 240
rec.leadTimeMinutes = 0
rec.leadTime = datetime.timedelta(days=10)
rec.validDate = datetime.datetime(2026, 3, 11, 12, 0)


Now notice the updated information in the is1 array.

In [14]:
print(f"{rec.is1 = }")

rec.is1 = array([        71,          1,       2026,          3,          1,
               12,          0, 2026030112,    1000008,        500,
              240,          0,        240,          0,          8,
                1,          0,          0,          0,          0,
                0,         32,         72,         69,         76,
               76,         79,         32,         87,         79,
               82,         76,         68,         33,         32,
               32,         32,         32,         32,         32,
               32,         32,         32,         32,         32,
               32,         32,         32,         32,         32,
               32,         32,         32,         32], dtype=int32)


**IMPORTANT:** The `leadTimeMinutes` attribute impacts, `leadTime` and `validDate`, **_but does not impact the value of "tau" in word 3 of the ID_**.

Once again, `validDate` is a read-only `@property` and setting it will raise an `AttributeError`.

In [15]:
# Capturing the error the notebook runs all cells.
try:
    rec.validDate = rec.refDate
except(AttributeError):
    print("Cannot set validDate")

Cannot set validDate


### TDLPACK ID

The TDLPACK ID (also known as the MOS-2000 Variable ID), in tdlpackio v2, is now its own class `TdlpackID`. The class provides various methods for formatting and `@property` for getting and setting ID components. The TDLPACK ID is a 4-word unsigned integer identification attribute.

| ID Word | Compontents |
| --- | --- |
| Word 1 | CCC FFF B DD |
| Word 2 | V LLLL UUUU |
| Word 3 | T RR O HH TAU |
| Word 4 | WXXXXYY I S G |

A complete description of the TDLPACK Varriable ID can be found in [TDL Office Note 00-1](https://www.weather.gov/media/mdl/TDL_OfficeNote00-1.pdf), Chapter 4.

The `TdlpackID` class provides a mechanism such that an instance can be linked to the `TdlpackRecord` instance it is contained within, thus allowing interaction between the 2 objects. The `TdlpackID` instance is accessible using the `.id` attribute.

In [16]:
print(rec.id)

[1000008, 500, 240, 0]


Internally, the ID information is stored as a dictionary in the private attribute `._id` where the dict keys are component names and the values are the integer component values with the expection of `thres`. The `thresh` comprised of `WXXXXYY` portion of word 4.

In [17]:
print(rec.id._id)
# Print each word
print(f"{rec.id.word1 = }")
print(f"{rec.id.word2 = }")
print(f"{rec.id.word3 = }")
print(f"{rec.id.word4 = }")

{'ccc': 1, 'fff': 0, 'b': 0, 'dd': 8, 'v': 0, 'llll': 0, 'uuuu': 500, 't': 0, 'rr': 0, 'o': 0, 'hh': 0, 'tau': 240, 'i': 0, 's': 0, 'g': 0, 'thresh': 0.0}
rec.id.word1 = 1000008
rec.id.word2 = 500
rec.id.word3 = 240
rec.id.word4 = 0


You can access the components as attributes of the ID object.

In [18]:
print(f"{rec.id.ccc = }")
print(f"{rec.id.fff = }")
print(f"{rec.id.dd = }")
print(f"{rec.id.uuuu = }")

rec.id.ccc = 1
rec.id.fff = 0
rec.id.dd = 8
rec.id.uuuu = 500


The `TdlpackID` class provides methods for providing the ID in various types and print formats.

In [19]:
print(f"{rec.id.to_dict() = }")
print(f"{rec.id.to_string() = }")
print(f"{rec.id.to_string(delim=":") = }")

rec.id.to_dict() = {'ccc': 1, 'fff': 0, 'b': 0, 'dd': 8, 'v': 0, 'llll': 0, 'uuuu': 500, 't': 0, 'rr': 0, 'o': 0, 'hh': 0, 'tau': 240, 'i': 0, 's': 0, 'g': 0, 'thresh': 0.0}
rec.id.to_string() = '001000008 000000500 000000240 0000000000'
rec.id.to_string(delim=":") = '001000008:000000500:000000240:0000000000'


And also supports the `__format__` protocol to allow format descriptors using f-strings.

| Format string | Description |
| --- | --- |
| `"basic"` or `"b"` | Four-word identifier. Each word is printed as a zero-padded integer field. |
| `"mos"` or `"m"` | MOS-style identifier consisting of the first three words followed by the ISG components and the threshold value formatted in scientific notation (`.0000e±00`). |
| `"parsed"` or `"p"` | Identifier parsed into its individual components as defined by the internal ID mapping. All components are printed as zero-padded integers except the threshold value, which is printed as a floating-point value with `F13.6` formatting. |

In [20]:
print(f"{rec.id:basic}")
print(f"{rec.id:mos}")
print(f"{rec.id:parsed}")

001000008 000000500 000000240 0000000000
001000008 000000500 000000240 000 .0000e+00
001 000 0 08 0 0000 0500 0 00 0 00 240 0 0 0      0.000000


### Accessing data

tdlpackio v2, in the same manner as grib2io, performs lazy loading of data. The data section of a TDLPACK record is only ever read from disk, unpacked, and return to the user when the `.data` attribute is referenced.

**_IMPORTANT:_** tdlpackio v2 returns gridded data in C/Python order -- (ny, nx). This is a **_major_** change from pytdlpack which provided data in Fortran order (nx, ny).

In [21]:
print(type(rec._data))
_ = rec.data
print(type(rec._data))

<class 'tdlpackio._tdlpackio.TdlpackRecordOnDiskArray'>
<class 'numpy.ndarray'>


The `TdlpackRecord` class provides a `.flush_data()` method to remove the data from the record object.

In [22]:
print(type(rec._data))
rec.flush_data()
print(type(rec._data))

<class 'numpy.ndarray'>
<class 'tdlpackio._tdlpackio.TdlpackRecordOnDiskArray'>


The `TdlpackRecord` class also provides basic statistics of the data: `.max`, `.min`, `.mean`, and `.median`.

In [23]:
print(f"{rec.min = }")
print(f"{rec.max = }")
print(f"{rec.mean = }")
print(f"{rec.median = }")

rec.min = np.float32(4902.0)
rec.max = np.float32(5939.0)
rec.mean = np.float32(5568.621)
rec.median = np.float32(5665.0)


### Closing an Open TDLPACK File

Closing a TDLPACK file is straightforward.

In [24]:
f.close()

## TDLPACK Sequential File (Vector Records)

We will now open a TDLPACK sequential vector (stations) file. The file contains METAR observations for Jan. 2017

In [25]:
f = tdlpackio.open("hre201701.sq")
print(f)

path = hre201701.sq
mode = rb
format = None
ra_template = None
name = /Users/ericengle/Repos/GitHub-NOAA/tdlpackio/demos/hre201701.sq
records = 23622
filetype = sequential
size = 55915400



Let's iterate over the first 35 records.

In [26]:
for rec in f[:35]:
    print(rec)

0:d=0000000000:STATION CALL LETTER RECORD:2892
1:d=2017010100:700002000 000000000 000000000 0000000000:  0-HR FCST: OBS TYPE                       
2:d=2017010100:400006000 000000000 000000000 0000000000:  0-HR FCST: LATITUDE                       
3:d=2017010100:400007000 000000000 000000000 0000000000:  0-HR FCST: LONGITUDE                      
4:d=2017010100:700001000 000000000 000000000 0000000000:  0-HR FCST: OBSERVATION TIME               
5:d=2017010100:702000000 000000000 000000000 0000000000:  0-HR FCST: OBS TEMPERATURE                
6:d=2017010100:703100000 000000000 000000000 0000000000:  0-HR FCST: OBS DEW POINT                  
7:d=2017010100:708500000 000000000 000000000 0000000000:  0-HR FCST: OBS WEATHER (PWX1)             
8:d=2017010100:708510000 000000000 000000000 0000000000:  0-HR FCST: OBS WEATHER (PWX2)             
9:d=2017010100:708520000 000000000 000000000 0000000000:  0-HR FCST: OBS WEATHER (PWX3)             
10:d=2017010100:708100000 000000000 00000000

The first record on the file is a `TdlpackStationRecord` object containing 2892 stations. It is important to note that when opening a station file, the stations are not unpacked until they are referenced. The stations are stored in the `._stations` private attribute. The station call letter record does define an `id` attribute. The ID is `[400001000, 0, 0, 0]` and is a "constant" ID to define a station record. This is not necessarily important for sequential files because there is not formal storage of the ID, but in a random-access file, the station record ID is used in the random-access file key records.

In [27]:
stations = f[0]
print(type(stations))
print(f"{stations.numberOfStations = }")
print(stations._stations)
print(f"{stations.id = }")

<class 'tdlpackio._tdlpackio.TdlpackStationRecord'>
stations.numberOfStations = 2892
None
stations.id = [400001000, 0, 0, 0]


When the `.stations` attribute is referenced, then the station data is read from file and "unpacked" into characters. Below are the first 25 stations on the record.

In [28]:
print(f"{stations.stations[:25] = }")

stations.stations[:25] = ['CABB', 'CABF', 'CABR', 'CABT', 'CACP', 'CACQ', 'CADS', 'CAFC', 'CAFY', 'CAHD', 'CAHK', 'CAHR', 'CAHW', 'CAJT', 'CAJW', 'CAMS', 'CAOH', 'CAOS', 'CAPR', 'CAQY', 'CARP', 'CAVA', 'CAWR', 'CBBC', 'CGMG']


Each succeeding `TdlpackRecord` object after a station record is linked to that station record, until a `TdlpackTrailerRecord` object is reached. This allows for the referencing data values using the station call letters. Lets demonstrate this using 6th record, `OBS TEMPERATURE` and get the value for `KBWI`.

In [29]:
rec = f[5]
print(rec)

5:d=2017010100:702000000 000000000 000000000 0000000000:  0-HR FCST: OBS TEMPERATURE                


In [30]:
station_id = "KBWI"
print(f"{station_id}: {rec["KBWI"]}")

KBWI: 47.0


Lets prove this further by finding the index for `KBWI` and use that value for for the data array index.

In [31]:
print(f"{station_id = }")
print(f"{stations.stations.index(station_id) = }")
print(rec.data[stations.stations.index(station_id)])

station_id = 'KBWI'
stations.stations.index(station_id) = 936
47.0


In [32]:
f.close()

## TDLPACK Random-Access File

We will now open a TDLPACK random-access gridded file. The file contains gridded constants.

In [33]:
f = tdlpackio.open("blend.analysisgrconst.co.ra")
print(f)

path = blend.analysisgrconst.co.ra
mode = rb
format = None
ra_template = None
name = /Users/ericengle/Repos/GitHub-NOAA/tdlpackio/demos/blend.analysisgrconst.co.ra
records = 7
filetype = random-access
master_key = {'version': 0, 'nids': 4, 'nwords': 5000, 'nkyrec': 1, 'maxent': 832, 'lastky': 2}
key_records = ({'nkeys': 7, 'prec_this_key': 1, 'prec_next_key': 99999999},)
size = 3220000



If this feels redundant, then good :) The goal for tdlpackio v2 is to make reading the various types of TDLPACK file feel the same. Here we are reading a random-access file, which is structurally very different than a sequential file. Printing the file object, you'll see a couple of attributes specific to random-access files.

| Attr | Description |
| --- | --- |
| `master_key` | Dictionary of the contents of the master key record |
| `key_records` | Tuple of dictionaries where each dict holds key record information |

A complete description of the structure of random-access files can be found in [TDL Office Note 00-1](https://www.weather.gov/media/mdl/TDL_OfficeNote00-1.pdf), Chapter 7.

In [34]:
for rec in f:
    print(rec)

0:d=1970010100:400350000 000000000 000000000 0000000000:  0-HR FCST:CONUS Land/Water Mask (v2.4.1)
1:d=1970010100:409350000 000000000 000000000 0000000000:  0-HR FCST:CONUS Terrain Elevation (v2.4)
2:d=1970010100:400359000 000000000 000000000 0000000000:  0-HR FCST:CONUS Clipping Mask (v2.4)
3:d=1970010100:400358000 000000000 000000000 0000000000:  0-HR FCST:CONUS Computational Mask (v2.4)
4:d=1970010100:400356000 000000000 000000000 0000000000:  0-HR FCST:CONUS Snow Clipping Mask (v2.4)
5:d=1970010100:409353000 000000000 000000000 0000000000:  0-HR FCST:CONUS Land Proximity Mask (v2.4.
6:d=1970010100:400351000 000000000 000000000 0000000000:  0-HR FCST:CONUS Binary Land/Water Mask (v2


Accessing TDLPACK records and data from a random-access file is the same as a sequential file.

## Predefined TDLPACK Grids

tdlpackio v2 provides predefined grids that are commonly used. These are available in `tdlpackio.grids.GRIDS` as `tdlpackio.grids.TdlpackGridDefinition` objects.

In [35]:
for k, v in tdlpackio.grids.GRIDS.items():
    print(f"{k}: {v}")

nbmak: TdlpackGridDefinition(mapProjection=5, nx=1649, ny=1105, latitudeLowerLeft=40.5301, longitudeLowerLeft=178.5713, standardLatitude=60.0, orientationLongitude=150.0, gridLength=2976.560059)
nbmco: TdlpackGridDefinition(mapProjection=3, nx=2345, ny=1597, latitudeLowerLeft=19.229, longitudeLowerLeft=126.2766, standardLatitude=25.0, orientationLongitude=95.0, gridLength=2539.702881)
nbmhi: TdlpackGridDefinition(mapProjection=7, nx=625, ny=561, latitudeLowerLeft=14.3515, longitudeLowerLeft=164.9695, standardLatitude=20.0, orientationLongitude=160.0, gridLength=2500.0)
nbmoc: TdlpackGridDefinition(mapProjection=7, nx=2517, ny=1817, latitudeLowerLeft=-30.4192, longitudeLowerLeft=230.0942, standardLatitude=20.0, orientationLongitude=360.0, gridLength=10000.0)
nbmpr: TdlpackGridDefinition(mapProjection=7, nx=353, ny=257, latitudeLowerLeft=16.828, longitudeLowerLeft=68.1954, standardLatitude=20.0, orientationLongitude=65.0, gridLength=1250.0)
nbmswp: TdlpackGridDefinition(mapProjection=7, 

The best practice method for getting these objects is to use the `tdlpackio.grids.get_grid()` function.

In [36]:
grid = tdlpackio.grids.get_grid("nbmco")
print(type(grid))

<class 'tdlpackio.grids.TdlpackGridDefinition'>


**IMPORTANT:** By default longitudes are specified in the `"mos2K"` convention of positive west longitude. The `get_grid()` accepts a kwarg `lon_format=` to provide longitudes in different formats.  See the table below.

| Format specifier | Definition |
| --- | ---|
| `"mos2k"` | West longitude positive (native storage format) |
| `"standard"` | East-positive, range [-180, 180] |
| `"0_360"` | East-positive, range [0, 360) |

In [37]:
for fmt in {"mos2k", "standard", "0_360"}:
    print(f"{fmt}: {tdlpackio.grids.get_grid("nbmco", lon_format="mos2k")}")

mos2k: TdlpackGridDefinition(mapProjection=3, nx=2345, ny=1597, latitudeLowerLeft=19.229, longitudeLowerLeft=126.2766, standardLatitude=25.0, orientationLongitude=95.0, gridLength=2539.702881)
0_360: TdlpackGridDefinition(mapProjection=3, nx=2345, ny=1597, latitudeLowerLeft=19.229, longitudeLowerLeft=126.2766, standardLatitude=25.0, orientationLongitude=95.0, gridLength=2539.702881)
standard: TdlpackGridDefinition(mapProjection=3, nx=2345, ny=1597, latitudeLowerLeft=19.229, longitudeLowerLeft=126.2766, standardLatitude=25.0, orientationLongitude=95.0, gridLength=2539.702881)


The `tdlpackio.grids.TdlpackGridDefinition` object provides a method for returning the grid definition information as an IS2 array.

In [38]:
print(grid.to_is2())

[      0       3    2345    1597  192290 1262766  950000 2539703  250000
       0       0       0       0       0       0       0       0       0
       0       0       0       0       0       0       0       0       0
       0       0       0       0       0       0       0       0       0
       0       0       0       0       0       0       0       0       0
       0       0       0       0       0       0       0       0       0]


## Creating a TdlpackRecord from Scratch
---

Lets create a TDLPACK record from scratch. By default, a `TdlpackRecord` objected is instantiated as a vector record. You can provide `type="grid"` to create a gridded record where you would need to provide the grid definition metadata. Or you can provide `grid=<string>` where the `<string>` is a predefined grid defintion found in `tdlpackio.grids.GRIDS`.

In [39]:
rec = tdlpackio.TdlpackRecord(grid="nbmco")
rec.refDate = datetime.datetime(2026, 2, 1, 12)
rec.id = [4210008,10,24,0]
rec.name = "GFS 10M WIND SPEED"
rec.data = np.random.rand(rec.nx*rec.ny).reshape(rec.shape)*75.0

In [40]:
print(rec)
print(f"{rec.min = }\n{rec.max = }\n{rec.mean = }\n{rec.median = }")

-1:d=2026020112:004210008 000000010 000000024 0000000000: 24-HR FCST:GFS 10M WIND SPEED
rec.min = np.float64(1.6823914344987756e-06)
rec.max = np.float64(74.99999488568835)
rec.mean = np.float64(37.492443278351324)
rec.median = np.float64(37.4719292926247)


You'll notice the "record number" in the first column of the print is `-1`. This informs the user that this instance of `TdlpackRecord` was not read from file.

### Packing a TdlpackRecord and Writing to File

Once a record is created and filled with data, you might want to pack and write to file. Lets first look at the packing information in section 4.

In [41]:
print(rec.__repr__())

Section 0: edition = 0
Section 1: sectionFlags = {'hasBitMapSection': 0, 'hasGridDefinitionSection': 1}
Section 1: year = 2026
Section 1: month = 2
Section 1: day = 1
Section 1: hour = 12
Section 1: minute = 0
Section 1: refDate = 2026-02-01 12:00:00
Section 1: id = 004210008 000000010 000000024 0000000000
Section 1: leadTime = 1 day, 0:00:00
Section 1: leadTimeHours = 24
Section 1: leadTimeMinutes = 0
Section 1: modelID = 0
Section 1: modelSequenceID = 0
Section 1: decScaleFactor = 0
Section 1: binScaleFactor = 0
Section 1: name = GFS 10M WIND SPEED              
Section 1: validDate = 2026-02-02 12:00:00
Section 1: duration = 0:00:00
Section 2: mapProjection = 3
Section 2: nx = 2345
Section 2: ny = 1597
Section 2: latitudeLowerLeft = 19.229
Section 2: longitudeLowerLeft = 126.2766
Section 2: orientationLongitude = 95.0
Section 2: gridLength = 2539.703
Section 2: standardLatitude = 25.0
Section 2: projParams = {'a': 6371229.0, 'b': 6371229.0, 'proj': 'lcc', 'lat_1': 25.0, 'lat_2': 25.

See the attributes prefixed with `Section 4:`.  You can also get section-specific attributes and values with `.attrs_by_section()` method. 

In [42]:
print(rec.attrs_by_section(4, values=True))

{'packingFlags': {'isVectorData': 0, 'packing': 0, 'packingOptions': 0, 'hasPrimaryMissingValue': 0, 'hasSecondaryMissingValue': 0}, 'numberOfPackedValues': np.int32(0), 'primaryMissingValue': np.int32(0), 'secondaryMissingValue': np.int32(0), 'overallMinValue': np.int32(0), 'numberOfGroups': np.int32(0)}


Before packing, we will want to set the decimal scale factor in order to preserve floating-point precision. **NOTE:** The decimal scale factor is stored in the `is1` array.

In [43]:
rec.decScaleFactor = 3

Once packed, the packed binary data live in the `.ipack` attribute which does not exist until data are packed.

In [44]:
print(hasattr(rec, "_ipack"))
rec.pack()
print(hasattr(rec, "_ipack"))

False
 ****NDG =65535 NOT LARGE ENOUGH IN PACKGP.  LMINPK IS INCREASED TO  31 FOR THIS FIELD.
True


Now lets write the packed record to a sequential file...

In [45]:
output = tdlpackio.open("wind_speed.sq", mode="w")
output.write(rec)
output.close()

Now lets read the new file...

In [46]:
with tdlpackio.open("wind_speed.sq") as fo:
    for rec in fo:
        print(rec)
        print(f"{rec.min = }\n{rec.max = }\n{rec.mean = }\n{rec.median = }")

0:d=2026020112:004210008 000000010 000000024 0000000000: 24-HR FCST:GFS 10M WIND SPEED
rec.min = np.float32(0.0)
rec.max = np.float32(75.0)
rec.mean = np.float32(37.492447)
rec.median = np.float32(37.472)


Note that the stats are **_very close_** to the original data, but not exact since data are decimal scaled.